In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

REPO_URL  = "https://github.com/irembuseozkose/qsvm-for-url-phishing"  # ← kendi repo linkin
REPO_DIR  = "/content/repo"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("📂 Çalışma dizini:", os.getcwd())
!ls

Cloning into '/content/repo'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 58 (delta 15), reused 52 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 479.42 KiB | 39.95 MiB/s, done.
Resolving deltas: 100% (15/15), done.
📂 Çalışma dizini: /content/repo
guides	notebooks  readme  requirements.t  src


In [6]:
# Drive'daki veri klasörünün yolunu ayarla
# Örnek: Drive'da "MyDrive/bitirme/data/processed/" altında .npy dosyaları varsa:
DRIVE_DATA = "/content/drive/MyDrive/Ag_Guvenligi_Projesi/data/processed"  # ← kendi yolun

# Kontrol
for f in ["X_train.npy", "X_test.npy", "y_train.npy", "y_test.npy"]:
    path = os.path.join(DRIVE_DATA, f)
    assert os.path.exists(path), f"❌ Bulunamadı: {path}"
    print(f"✅ {f}")

print("\n✅ Tüm veri dosyaları mevcut.")

✅ X_train.npy
✅ X_test.npy
✅ y_train.npy
✅ y_test.npy

✅ Tüm veri dosyaları mevcut.


In [7]:
!pip install qiskit qiskit-aer pennylane -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 142.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 143.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 117.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 158.1 MB/s eta 0:00:0000:01


In [8]:
import sys
sys.path.insert(0, REPO_DIR)

from src.models.quantum_kernel import QuantumKernel
from src.models.qsvm_model     import QSVM
from src.models.pso_qsvm       import PSOQSVM
from src.models.svm_model      import ClassicalSVM

import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

print("✅ Modüller yüklendi.")

✅ Modüller yüklendi.


In [ ]:
X_train = np.load(os.path.join(DRIVE_DATA, "X_train.npy"))
X_test  = np.load(os.path.join(DRIVE_DATA, "X_test.npy"))
y_train = np.load(os.path.join(DRIVE_DATA, "y_train.npy"))
y_test  = np.load(os.path.join(DRIVE_DATA, "y_test.npy"))

X_all_raw = np.concatenate([X_train, X_test], axis=0)
y_all     = np.concatenate([y_train, y_test], axis=0)

print(f"Orijinal özellik sayısı : {X_all_raw.shape[1]}")

# ──────────────────────────────────
# PCA ile özellik indirgeme
# 8 özellik → ceil(log2(8)) = 3 qubit → dim=8
# ──────────────────────────────────
N_COMPONENTS = 8

pca = PCA(n_components=N_COMPONENTS, whiten=True, random_state=42)
X_all_pca = pca.fit_transform(X_all_raw)

print(f"PCA sonrası             : {X_all_pca.shape[1]} özellik")
print(f"Açıklanan varyans       : {pca.explained_variance_ratio_.sum():.4f} ({pca.explained_variance_ratio_.sum()*100:.1f}%)")
print(f"Qubit sayısı            : {int(np.ceil(np.log2(N_COMPONENTS)))} (önceki: 5)")

# MinMax + L2 normalizasyon
scaler = MinMaxScaler()
X_all_scaled = scaler.fit_transform(X_all_pca)
X_all = normalize(X_all_scaled, norm='l2')

print(f"\nToplam veri : {X_all.shape}")
print(f"Sınıflar    : {np.unique(y_all)}")
print(f"Dağılım     : {dict(zip(*np.unique(y_all, return_counts=True)))}")

Toplam veri : (100000, 21)
Sınıflar    : [0 1 2 3]
Dağılım     : {np.int64(0): np.int64(25000), np.int64(1): np.int64(25000), np.int64(2): np.int64(25000), np.int64(3): np.int64(25000)}


In [ ]:
pca_full = PCA(random_state=42).fit(X_all_raw)
cumvar   = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(cumvar)+1), cumvar, "o-", color="steelblue", markersize=5)
ax.axhline(y=cumvar[N_COMPONENTS-1], color="coral", linestyle="--",
           label=f"{N_COMPONENTS} bileşen = {cumvar[N_COMPONENTS-1]*100:.1f}% varyans")
ax.axvline(x=N_COMPONENTS, color="coral", linestyle="--", alpha=0.5)
ax.set_xlabel("Bileşen Sayısı")
ax.set_ylabel("Kümülatif Açıklanan Varyans")
ax.set_title("PCA — Kümülatif Varyans")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("pca_variance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
N_SPLITS     = 3   # 5'ten 3'e düşürüldü
RANDOM_STATE = 42
CLASS_NAMES  = ["benign", "defacement", "malware", "phishing"]

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
folds = list(skf.split(X_all, y_all))

print(f"K-Fold ayarları:")
print(f"  Fold sayısı    : {N_SPLITS}")
print(f"  Random state   : {RANDOM_STATE}")
print(f"  Stratified     : Evet")
print(f"  PCA bileşen    : {N_COMPONENTS}")
print(f"  Qubit sayısı   : {int(np.ceil(np.log2(N_COMPONENTS)))}\n")

for i, (train_idx, test_idx) in enumerate(folds):
    print(f"  Fold {i+1}: train={len(train_idx):,}  test={len(test_idx):,}  "
          f"test dağılım={dict(zip(*np.unique(y_all[test_idx], return_counts=True)))}")

K-Fold ayarları:
  Fold sayısı    : 5
  Random state   : 42
  Stratified     : Evet

  Fold 1: train=80,000  test=20,000  test dağılım={np.int64(0): np.int64(5000), np.int64(1): np.int64(5000), np.int64(2): np.int64(5000), np.int64(3): np.int64(5000)}
  Fold 2: train=80,000  test=20,000  test dağılım={np.int64(0): np.int64(5000), np.int64(1): np.int64(5000), np.int64(2): np.int64(5000), np.int64(3): np.int64(5000)}
  Fold 3: train=80,000  test=20,000  test dağılım={np.int64(0): np.int64(5000), np.int64(1): np.int64(5000), np.int64(2): np.int64(5000), np.int64(3): np.int64(5000)}
  Fold 4: train=80,000  test=20,000  test dağılım={np.int64(0): np.int64(5000), np.int64(1): np.int64(5000), np.int64(2): np.int64(5000), np.int64(3): np.int64(5000)}
  Fold 5: train=80,000  test=20,000  test dağılım={np.int64(0): np.int64(5000), np.int64(1): np.int64(5000), np.int64(2): np.int64(5000), np.int64(3): np.int64(5000)}


In [11]:
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy" : accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall"   : recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1"       : f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

print("✅ compute_metrics tanımlandı.")

✅ compute_metrics tanımlandı.


In [ ]:
svm_fold_metrics = []
svm_fold_preds   = []

print("⏳ Klasik SVM — 5-Fold Cross-Validation başlıyor...\n")

for i, (train_idx, test_idx) in enumerate(folds):
    t0 = time.time()

    X_tr, X_te = X_all[train_idx], X_all[test_idx]
    y_tr, y_te = y_all[train_idx], y_all[test_idx]

    svm = ClassicalSVM()
    svm.fit(X_tr, y_tr)
    y_pred = svm.predict(X_te)

    metrics = compute_metrics(y_te, y_pred)
    svm_fold_metrics.append(metrics)
    svm_fold_preds.append((y_te, y_pred))

    elapsed = time.time() - t0
    print(f"  Fold {i+1}: Acc={metrics['Accuracy']:.4f}  "
          f"P={metrics['Precision']:.4f}  R={metrics['Recall']:.4f}  "
          f"F1={metrics['F1']:.4f}  ({elapsed:.1f}s)")

print("\n✅ Klasik SVM K-Fold tamamlandı.")
#12dk sürdü, fold başına ortalama 2.4dk

⏳ Klasik SVM — 5-Fold Cross-Validation başlıyor...

  Fold 1: Acc=0.8331  P=0.8371  R=0.8331  F1=0.8325  (145.4s)
  Fold 2: Acc=0.8386  P=0.8434  R=0.8386  F1=0.8379  (146.8s)
  Fold 3: Acc=0.8373  P=0.8421  R=0.8373  F1=0.8366  (149.9s)
  Fold 4: Acc=0.8326  P=0.8379  R=0.8325  F1=0.8319  (145.1s)
  Fold 5: Acc=0.8339  P=0.8385  R=0.8339  F1=0.8334  (148.7s)

✅ Klasik SVM K-Fold tamamlandı.


In [ ]:
qsvm_fold_metrics = []
qsvm_fold_preds   = []

print("⏳ QSVM — 5-Fold Cross-Validation başlıyor...")
print("   (Her fold uzun sürebilir — quantum kernel hesaplaması nedeniyle)\n")

for i, (train_idx, test_idx) in enumerate(folds):
    t0 = time.time()

    X_tr, X_te = X_all[train_idx], X_all[test_idx]
    y_tr, y_te = y_all[train_idx], y_all[test_idx]

    qkernel = QuantumKernel()
    qsvm = QSVM(quantum_kernel=qkernel)
    qsvm.fit(X_tr, y_tr)
    y_pred = qsvm.predict(X_te)

    metrics = compute_metrics(y_te, y_pred)
    qsvm_fold_metrics.append(metrics)
    qsvm_fold_preds.append((y_te, y_pred))

    elapsed = time.time() - t0
    print(f"  Fold {i+1}: Acc={metrics['Accuracy']:.4f}  "
          f"P={metrics['Precision']:.4f}  R={metrics['Recall']:.4f}  "
          f"F1={metrics['F1']:.4f}  ({elapsed:.1f}s)")

print("\n✅ QSVM K-Fold tamamlandı.")

⏳ QSVM — 5-Fold Cross-Validation başlıyor...
   (Her fold uzun sürebilir — quantum kernel hesaplaması nedeniyle)

>> QuantumKernel: 21 features → 5 qubits (dim=32)


: 

In [ ]:
pso_fold_metrics = []
pso_fold_preds   = []
pso_best_Cs      = []

print("⏳ PSO-QSVM — 5-Fold Cross-Validation başlıyor...")
print("   (En uzun süren model — PSO + quantum kernel)\n")

for i, (train_idx, test_idx) in enumerate(folds):
    t0 = time.time()

    X_tr, X_te = X_all[train_idx], X_all[test_idx]
    y_tr, y_te = y_all[train_idx], y_all[test_idx]

    qkernel = QuantumKernel()
    pso = PSOQSVM(
        quantum_kernel=qkernel,
        n_particles=10,
        n_iters=20,
        C_min=1e-3,
        C_max=1e3,
        cv_splits=3,
        random_state=42,
    )
    pso.fit(X_tr, y_tr)
    y_pred = pso.predict(X_te)

    metrics = compute_metrics(y_te, y_pred)
    pso_fold_metrics.append(metrics)
    pso_fold_preds.append((y_te, y_pred))
    pso_best_Cs.append(pso.best_C)

    elapsed = time.time() - t0
    print(f"  Fold {i+1}: Acc={metrics['Accuracy']:.4f}  "
          f"P={metrics['Precision']:.4f}  R={metrics['Recall']:.4f}  "
          f"F1={metrics['F1']:.4f}  C={pso.best_C:.4f}  ({elapsed:.1f}s)")

print(f"\n✅ PSO-QSVM K-Fold tamamlandı.")
print(f"   Bulunan C değerleri: {[round(c, 4) for c in pso_best_Cs]}")

In [ ]:
models_data = {
    "Klasik SVM": svm_fold_metrics,
    "QSVM":       qsvm_fold_metrics,
    "PSO-QSVM":   pso_fold_metrics,
}

for model_name, fold_list in models_data.items():
    df_folds = pd.DataFrame(fold_list, index=[f"Fold {i+1}" for i in range(N_SPLITS)])
    print(f"\n{'='*55}")
    print(f"{model_name} — Fold Bazında Sonuçlar")
    print(f"{'='*55}")
    print(df_folds.round(4).to_string())
    print(f"\nOrtalama : {df_folds.mean().round(4).to_dict()}")
    print(f"Std      : {df_folds.std().round(4).to_dict()}")

In [ ]:
summary_rows = {}

for model_name, fold_list in models_data.items():
    df_folds = pd.DataFrame(fold_list)
    row = {}
    for col in df_folds.columns:
        mean = df_folds[col].mean()
        std  = df_folds[col].std()
        row[col] = f"{mean:.4f} ± {std:.4f}"
    summary_rows[model_name] = row

df_summary = pd.DataFrame(summary_rows).T
print("\n" + "="*70)
print("K-FOLD CROSS-VALIDATION ÖZET SONUÇLARI (Ortalama ± Std)")
print("="*70)
print(df_summary.to_string())

In [ ]:
metrics_list = ["Accuracy", "Precision", "Recall", "F1"]
model_names  = ["Klasik SVM", "QSVM", "PSO-QSVM"]
colors       = ["steelblue", "coral", "mediumseagreen"]

means = {}
stds  = {}
for model_name, fold_list in models_data.items():
    df_f = pd.DataFrame(fold_list)
    means[model_name] = df_f.mean()
    stds[model_name]  = df_f.std()

x     = np.arange(len(metrics_list))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))

for idx, (model, color) in enumerate(zip(model_names, colors)):
    ax.bar(
        x + (idx - 1) * width,
        [means[model][m] for m in metrics_list],
        width,
        yerr=[stds[model][m] for m in metrics_list],
        label=model,
        color=color,
        capsize=4,
        edgecolor="white",
        linewidth=0.5,
    )

ax.set_xticks(x)
ax.set_xticklabels(metrics_list, fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Skor", fontsize=12)
ax.set_title("5-Fold Cross-Validation Sonuçları (Ortalama ± Std)", fontsize=14)
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("kfold_comparison_bar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

fold_labels = [f"Fold {i+1}" for i in range(N_SPLITS)]

for model_name, fold_list, color, marker in zip(
    model_names,
    [svm_fold_metrics, qsvm_fold_metrics, pso_fold_metrics],
    colors,
    ["o", "s", "D"],
):
    accs = [m["Accuracy"] for m in fold_list]
    ax.plot(fold_labels, accs, marker=marker, label=model_name,
            color=color, linewidth=2, markersize=8)

ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Fold Bazında Accuracy Karşılaştırması", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_ylim(0.75, 0.95)

plt.tight_layout()
plt.savefig("kfold_accuracy_line.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
all_preds = {
    "Klasik SVM": svm_fold_preds,
    "QSVM":       qsvm_fold_preds,
    "PSO-QSVM":   pso_fold_preds,
}

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (model_name, preds_list) in zip(axes, all_preds.items()):
    y_true_all = np.concatenate([p[0] for p in preds_list])
    y_pred_all = np.concatenate([p[1] for p in preds_list])

    cm = confusion_matrix(y_true_all, y_pred_all)
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, colorbar=False, cmap="Blues"
    )
    ax.set_title(f"{model_name}\n(Tüm Foldlar Birleşik)", fontsize=13)
    plt.setp(ax.get_xticklabels(), rotation=20)

plt.tight_layout()
plt.savefig("kfold_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
for model_name, preds_list in all_preds.items():
    y_true_all = np.concatenate([p[0] for p in preds_list])
    y_pred_all = np.concatenate([p[1] for p in preds_list])

    print("=" * 60)
    print(f"{model_name} — Classification Report (5-Fold Birleşik)")
    print("=" * 60)
    print(classification_report(y_true_all, y_pred_all, target_names=CLASS_NAMES))

In [ ]:
f1_data = {
    name: [m["F1"] for m in folds_list]
    for name, folds_list in models_data.items()
}

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(
    f1_data.values(),
    labels=f1_data.keys(),
    patch_artist=True,
    widths=0.5,
)

for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel("F1 Score (macro)", fontsize=12)
ax.set_title("5-Fold CV — F1 Score Dağılımı", fontsize=14)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("kfold_f1_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/bitirme/data/results"  # ← kendi yolun
os.makedirs(SAVE_DIR, exist_ok=True)

# Fold bazında detaylı sonuçlar
all_fold_rows = []
for model_name, fold_list in models_data.items():
    for i, metrics in enumerate(fold_list):
        row = {"Model": model_name, "Fold": i + 1}
        row.update(metrics)
        all_fold_rows.append(row)

df_all_folds = pd.DataFrame(all_fold_rows)
df_all_folds.to_csv(os.path.join(SAVE_DIR, "kfold_detailed_results.csv"), index=False)

# Özet sonuçlar
summary_numeric = {}
for model_name, fold_list in models_data.items():
    df_f = pd.DataFrame(fold_list)
    summary_numeric[model_name] = {
        "Accuracy_mean": df_f["Accuracy"].mean(),
        "Accuracy_std":  df_f["Accuracy"].std(),
        "Precision_mean": df_f["Precision"].mean(),
        "Precision_std":  df_f["Precision"].std(),
        "Recall_mean": df_f["Recall"].mean(),
        "Recall_std":  df_f["Recall"].std(),
        "F1_mean": df_f["F1"].mean(),
        "F1_std":  df_f["F1"].std(),
    }

df_summary_num = pd.DataFrame(summary_numeric).T
df_summary_num.to_csv(os.path.join(SAVE_DIR, "kfold_summary_results.csv"))

# Tahminleri kaydet
for model_name, preds_list in all_preds.items():
    y_true_all = np.concatenate([p[0] for p in preds_list])
    y_pred_all = np.concatenate([p[1] for p in preds_list])
    safe_name  = model_name.lower().replace(" ", "_").replace("-", "_")
    np.save(os.path.join(SAVE_DIR, f"y_true_kfold_{safe_name}.npy"), y_true_all)
    np.save(os.path.join(SAVE_DIR, f"y_pred_kfold_{safe_name}.npy"), y_pred_all)

# Grafikleri de Drive'a kopyala
import shutil
for png in ["kfold_comparison_bar.png", "kfold_accuracy_line.png",
            "kfold_confusion_matrices.png", "kfold_f1_boxplot.png"]:
    if os.path.exists(png):
        shutil.copy(png, os.path.join(SAVE_DIR, png))

print("✅ Tüm sonuçlar Drive'a kaydedildi:")
print(f"   {SAVE_DIR}/")
!ls -la {SAVE_DIR}